# Génération du CSV de features (v2)

Ce notebook calcule un jeu de features numériques par image à partir du pipeline de preprocessing de Noura :

- **Images** chargées depuis `COVID-19_Radiography_Dataset_CLAHE` (radios après masque + égalisation + CLAHE), structure `<classe>/<image>.png`.
- **Masques** : vrais masques binaires chargés directement depuis `COVID-19_Radiography_Dataset` (structure `<classe>/masks/<image>.png`), réalignés sur l'image CLAHE si besoin puis appliqués.

**Features calculées** : `pixel_mean`, `pixel_std`, `lum_interieur_masque`, `lum_exterieur_masque`, `surface_masque`, `variance_laplacien`.

Sortie : `C:\Users\Maxime\Documents\Liora_Covid\data\processed\features_v2.csv`.

In [1]:
# Chargement des bibliothèques
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

In [8]:
# Chemins d'accès
clahe_path = Path(r"../../../COVID-19_Radiography_Dataset_CLAHE")
masks_path = Path(r"../../../COVID-19_Radiography_Dataset")
output_path = Path(r"../../../processed")
output_csv = Path(r"../../../processed/features_v2.csv")

classes = ["COVID", "Normal", "Lung_Opacity", "Viral Pneumonia"]

print("Dossier CLAHE   :", clahe_path, "->", clahe_path.exists())
print("Dossier masques :", masks_path, "->", masks_path.exists())
print("Dossier output :", output_path, "->", output_path.exists())

Dossier CLAHE   : ../../../COVID-19_Radiography_Dataset_CLAHE -> True
Dossier masques : ../../../COVID-19_Radiography_Dataset -> True
Dossier output : ../../../processed -> True


## Fonction d'extraction des features

Pour chaque image : chargement de l'image CLAHE + du vrai masque binaire (`masks/`), réalignement du masque sur l'image si nécessaire, binarisation (`> 0`) et calcul des features.

In [3]:
def extract_features_for_class(classe, clahe_path, masks_path, print_every=500, counter_start=0):
    """Extrait les features de chaque image d'une classe.

    - Image CLAHE : <clahe_path>/<classe>/<name>.png
    - Masque      : <masks_path>/<classe>/masks/<name>.png
      (vrai masque binaire du dataset d'origine)

    Retourne (liste de dict, compteur global mis a jour).
    """
    clahe_dir = clahe_path / classe

    rows = []
    count = counter_start

    for img_path in sorted(clahe_dir.glob("*.png")):

        clahe_img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)

        mask_path = masks_path / classe / "masks" / img_path.name
        mask_img  = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        if clahe_img is None:
            print(f"  [SKIP] image CLAHE illisible : {img_path.name}")
            continue
        if mask_img is None:
            print(f"  [SKIP] masque manquant : {img_path.name}")
            continue

        # Alignement du masque sur l'image CLAHE si les tailles different (masques 256, images 299)
        if mask_img.shape != clahe_img.shape:
            mask_img = cv2.resize(mask_img, (clahe_img.shape[1], clahe_img.shape[0]),
                                  interpolation=cv2.INTER_NEAREST)

        # Masque binaire (vrai masque du dataset d'origine)
        mask_bin = mask_img > 0

        pixel_mean = float(clahe_img.mean())
        pixel_std  = float(clahe_img.std())

        surface_masque = float(mask_bin.sum()) / float(mask_bin.size)

        lum_interieur_masque = float(clahe_img[mask_bin].mean())  if mask_bin.any()   else float("nan")
        lum_exterieur_masque = float(clahe_img[~mask_bin].mean()) if (~mask_bin).any() else float("nan")

        variance_laplacien = float(cv2.Laplacian(clahe_img, cv2.CV_64F).var())

        rows.append({
            "filename"             : img_path.name,
            "classe"               : classe,
            "pixel_mean"           : round(pixel_mean, 4),
            "pixel_std"            : round(pixel_std, 4),
            "lum_interieur_masque" : round(lum_interieur_masque, 4),
            "lum_exterieur_masque" : round(lum_exterieur_masque, 4),
            "surface_masque"       : round(surface_masque, 6),
            "variance_laplacien"   : round(variance_laplacien, 4),
        })

        count += 1
        if count % print_every == 0:
            print(f"  {count} images traitees")

    return rows, count

## Extraction pour les 4 classes

In [4]:
all_rows = []
count = 0

for classe in classes:
    print(f"=== {classe} ===")
    rows, count = extract_features_for_class(classe, clahe_path, masks_path,
                                             print_every=500, counter_start=count)
    all_rows.extend(rows)
    print(f"  {len(rows)} images traitees pour la classe {classe}\n")

print(f"Total lignes extraites : {len(all_rows)}")

=== COVID ===
  500 images traitees
  1000 images traitees
  1500 images traitees
  2000 images traitees
  2500 images traitees
  3000 images traitees
  3373 images traitees pour la classe COVID

=== Normal ===
  3500 images traitees
  4000 images traitees
  4500 images traitees
  5000 images traitees
  5500 images traitees
  6000 images traitees
  6500 images traitees
  7000 images traitees
  7500 images traitees
  8000 images traitees
  8500 images traitees
  9000 images traitees
  9500 images traitees
  10000 images traitees
  10500 images traitees
  11000 images traitees
  11500 images traitees
  12000 images traitees
  12500 images traitees
  13000 images traitees
  13500 images traitees
  10136 images traitees pour la classe Normal

=== Lung_Opacity ===
  14000 images traitees
  14500 images traitees
  15000 images traitees
  15500 images traitees
  16000 images traitees
  16500 images traitees
  17000 images traitees
  17500 images traitees
  18000 images traitees
  18500 images

## Construction du DataFrame et sauvegarde

In [9]:
# Creation du dossier de sortie si necessaire
output_csv.parent.mkdir(parents=True, exist_ok=True)

df_features = pd.DataFrame(all_rows)

# Ordre des colonnes
df_features = df_features[[
    "filename", "classe",
    "pixel_mean", "pixel_std",
    "lum_interieur_masque", "lum_exterieur_masque",
    "surface_masque", "variance_laplacien",
]]

df_features.to_csv(output_csv, index=False)
print("CSV sauvegarde :", output_csv)
print("Shape :", df_features.shape)
df_features.head()

CSV sauvegarde : ../../../processed/features_v2.csv
Shape : (20835, 8)


,filename,classe,pixel_mean,pixel_std,lum_interieur_masque,lum_exterieur_masque,surface_masque,variance_laplacien
0,COVID-1.png,COVID,42.2313,63.8117,144.7131,10.9031,0.234125,1779.5722
1,COVID-10.png,COVID,45.0521,65.8467,140.9980,10.8718,0.262670,2205.6286
2,COVID-1000.png,COVID,42.3115,63.1703,139.1298,10.8428,0.245299,1697.4233
3,COVID-1001.png,COVID,29.9235,52.1309,139.1083,10.9089,0.148320,1661.2817
4,COVID-1002.png,COVID,65.1165,75.3402,138.8807,10.7731,0.424201,2473.2551


## Distribution par classe et statistiques descriptives

In [10]:
print("-- Repartition par classe --")
display(df_features["classe"].value_counts().to_frame("n_images"))

-- Repartition par classe --


,n_images
classe,
Normal,10136
Lung_Opacity,5988
COVID,3373
Viral Pneumonia,1338


In [ ]:
print("-- Statistiques descriptives (features numeriques) --")
display(df_features.describe())

-- Statistiques descriptives (features numeriques) --


,pixel_mean,pixel_std,lum_interieur_masque,lum_exterieur_masque,surface_masque,variance_laplacien
count,20835.000000,20835.000000,20835.000000,20835.000000,20835.000000,20835.000000
mean,41.702772,62.714614,140.141323,10.879673,0.238575,1992.970682
std,7.838310,6.294955,3.087672,0.047459,0.061345,404.476972
min,14.667400,24.352300,121.003800,10.563700,0.028959,234.472300
25%,36.248200,58.822550,138.503600,10.853000,0.195881,1707.533100
50%,41.305000,63.223100,140.647700,10.884600,0.235031,2006.759000
75%,46.693250,67.131950,142.276450,10.913100,0.277066,2283.855750
max,84.562700,81.328200,151.300500,10.997800,0.579781,4552.092500
